# **Build A Transformer-based Translator**

Notebook from the book Build a Text to Image Generator from Scratch by Mark Liu, published by Manning Publications. See https://mng.bz/vZem and https://github.com/markhliu for details.

Be sure to set the runtime type of this colab notebook to GPU

German2English translator based on 29K German2englih pairs


In [20]:
# clone the book's GitHub repository
colab=True
if colab:
    !git clone https://github.com/markhliu/txt2img
#! pip install spacy

Cloning into 'txt2img'...
remote: Enumerating objects: 436, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 436 (delta 81), reused 52 (delta 52), pack-reused 351 (from 1)
Receiving objects: 100% (436/436), 88.66 MiB | 28.93 MiB/s, done.
Resolving deltas: 100% (213/213), done.


In [31]:
# allow access to local modules and files in the book's repository
import sys
if colab:
    sys.path.append("/content/txt2img") # in Colab
else:
    sys.path.append("./txt2img")
# 

## Downloading the training dataset

In [36]:
import requests, os, tarfile

url=("https://raw.githubusercontent.com/neychev/"
     "small_DL_repo/master/datasets/Multi30k/training.tar.gz")  # The URL to download the training dataset
os.makedirs("files", exist_ok=True)
if not os.path.exists("files/training.tar.gz"):  # Download the dataset to your computer
    fb1=requests.get(url)
    with open("files/training.tar.gz","wb") as f:
        f.write(fb1.content)
train=tarfile.open('files/training.tar.gz')  # Unzip the file
train.extractall('files')  # Place content in the /files/ folder
train.close()



In [37]:
with open("files/train.de", 'rb') as fb:
    trainde = fb.readlines()
with open("files/train.en", 'rb') as fb:
    trainen = fb.readlines()
trainde=[i.decode("utf-8").strip() for i in trainde]
trainen=[i.decode("utf-8").strip() for i in trainen]


In [39]:
from pprint import pprint

print(f"the length of the list trainde is {len(trainde)}")
print(f"the length of the list trainen is {len(trainen)}")
print(f"\n the first five elements of the list trainde (German) are")
pprint(trainde[:5])
print(f"\n the first five elements of the list trainen (English) are")
pprint(trainen[:5])


the length of the list trainde is 29001
the length of the list trainen is 29001

 the first five elements of the list trainde (German) are
['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.',
 'Ein kleines Mädchen klettert in ein Spielhaus aus Holz.',
 'Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.',
 'Zwei Männer stehen am Herd und bereiten Essen zu.']

 the first five elements of the list trainen (English) are
['Two young, White males are outside near many bushes.',
 'Several men in hard hats are operating a giant pulley system.',
 'A little girl climbing into a wooden playhouse.',
 'A man in a blue shirt is standing on a ladder cleaning a window.',
 'Two men are at the stove preparing food.']


### Use the tokenizers in the Spacy library to convert both German and Englishphrases into tokens. 

In [41]:
import os, spacy

try:
    de_tokenizer = spacy.load("de_core_news_sm")  # Try to load the German tokenizer
except IOError:
    os.system("python -m spacy download de_core_news_sm")  #B
    de_tokenizer = spacy.load("de_core_news_sm")
try:
    en_tokenizer = spacy.load("en_core_web_sm")  # Try to load the English tokenizer 
except IOError:
    os.system("python -m spacy download en_core_web_sm")  #D
    en_tokenizer = spacy.load("en_core_web_sm")


### Use the two tokenizers to convert the first German phrase and the first Englishphrase into tokens and print them out:

In [46]:
tokenized_de= [tok.text for tok in
              de_tokenizer.tokenizer(trainde[0])]
tokenized_en=[tok.text for tok in
              en_tokenizer.tokenizer(trainen[0])]
print(trainde[0], tokenized_de)
print(trainen[0],tokenized_en)


Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche. ['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
Two young, White males are outside near many bushes. ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


## Creating a dictionary to map English tokens to indices and viceversa

In [72]:
from collections import Counter

en_tokens=[["BOS"]+[tok.text for tok in en_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainen]  #Add BOS and EOS at the beginning and end of each phrase
PAD=0
UNK=1
word_count=Counter()
for sentence in en_tokens:
    for word in sentence:
        word_count[word]+=1
frequency=word_count.most_common(50000)
print("5 most frequent tokens:", frequency[:5])

total_en_words=len(frequency)+2
# a dictionary mapping tokens to indexes


keys = list(range(1,10))

en_word_dict={w[0]:idx+2 for idx,w in enumerate(frequency)}  #Assign an index to each unique token
en_word_dict["PAD"]=PAD
en_word_dict["UNK"]=UNK  #The padding token and unknown tokens are assigned indexes 0 and 1, respectively

print("Token2Index:", {k: v for k, v in en_word_dict.items() if v in keys})

# another dictionary to map indexes to tokens
en_idx_dict={v:k for k,v in en_word_dict.items()}  #A dictionary to map indexes back to tokens


print("Index2Token:", {k: en_idx_dict[k] for k in keys} )





5 most frequent tokens: [('a', 31707), ('BOS', 29001), ('EOS', 29001), ('.', 27623), ('A', 17458)]
Token2Index: {'a': 2, 'BOS': 3, 'EOS': 4, '.': 5, 'A': 6, 'in': 7, 'the': 8, 'on': 9, 'UNK': 1}
Index2Token: {1: 'UNK', 2: 'a', 3: 'BOS', 4: 'EOS', 5: '.', 6: 'A', 7: 'in', 8: 'the', 9: 'on'}


#### From tokens to indices and viceversa

In [74]:

print("List of tokens:", tokenized_en)
enidx=[en_word_dict.get(i,UNK) for i in tokenized_en]
print("List of indices:", enidx)


List of tokens ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']
List of indices [19, 25, 15, 1165, 804, 17, 57, 84, 334, 1329, 5]


In [8]:
entokens=[en_idx_dict.get(i,"UNK") for i in enidx]
print(entokens)
en_phrase=" ".join(entokens)
for x in '''?:;.,'("-!&)%''':
    en_phrase=en_phrase.replace(f" {x}",f"{x}")
print(en_phrase)


['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']
Two young, White males are outside near many bushes.


## Creating a dictionary to map German tokens to indices and viceversa


In [75]:
de_tokens= [["BOS"]+[tok.text for tok in de_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainde]  # Add BOS and EOS at the beginning and end of each phrase
de_word_count=Counter()
for sentence in de_tokens:
    for word in sentence:
        de_word_count[word]+=1
defrequency=de_word_count.most_common(50000)
total_de_words=len(defrequency)+2
de_word_dict={w[0]:idx+2 for idx,w in enumerate(defrequency)}  # Assign an index to each unique token
de_word_dict["PAD"]=PAD
de_word_dict["UNK"]=UNK  # The padding token and unknown tokens are assigned indexes 0 and 1, respectively 
de_idx_dict={v:k for k,v in de_word_dict.items()}  # A dictionary to map indexes back to tokens



#### From tokens to indices and viceversa

In [76]:
print(tokenized_de)
deidx=[de_word_dict.get(i,UNK) for i in tokenized_de]
print(deidx)


['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
[21, 85, 257, 31, 87, 22, 94, 7, 16, 112, 5497, 3161, 4]


In [77]:
detokens=[de_idx_dict.get(i,"UNK") for i in deidx]
print(detokens)
de_phrase=" ".join(detokens)
for x in '''?:;.,'("-!&)%''':
    de_phrase=de_phrase.replace(f" {x}",f"{x}")
print(de_phrase)


['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


### Convert all English phrases to their numerical representations and do the same for German phrases:

In [12]:
out_en_ids=[[en_word_dict.get(w,UNK) for w in s]
            for s in en_tokens]
out_de_ids=[[de_word_dict.get(w,UNK) for w in s]
            for s in de_tokens]
sorted_ids=sorted(range(len(out_de_ids)),
                  key=lambda x:len(out_de_ids[x]))
out_de_ids=[out_de_ids[x] for x in sorted_ids]
out_en_ids=[out_en_ids[x] for x in sorted_ids]


### Put the numerical representations into batches for training

In [79]:
import numpy as np

batch_size=128
idx_list=np.arange(0,len(de_tokens),batch_size)
np.random.shuffle(idx_list)

batch_indexs=[]
for idx in idx_list:
    batch_indexs.append(np.arange(idx,min(len(de_tokens),
                                          idx+batch_size)))

print(batch_indexs[:2])

[array([23552, 23553, 23554, 23555, 23556, 23557, 23558, 23559, 23560,
       23561, 23562, 23563, 23564, 23565, 23566, 23567, 23568, 23569,
       23570, 23571, 23572, 23573, 23574, 23575, 23576, 23577, 23578,
       23579, 23580, 23581, 23582, 23583, 23584, 23585, 23586, 23587,
       23588, 23589, 23590, 23591, 23592, 23593, 23594, 23595, 23596,
       23597, 23598, 23599, 23600, 23601, 23602, 23603, 23604, 23605,
       23606, 23607, 23608, 23609, 23610, 23611, 23612, 23613, 23614,
       23615, 23616, 23617, 23618, 23619, 23620, 23621, 23622, 23623,
       23624, 23625, 23626, 23627, 23628, 23629, 23630, 23631, 23632,
       23633, 23634, 23635, 23636, 23637, 23638, 23639, 23640, 23641,
       23642, 23643, 23644, 23645, 23646, 23647, 23648, 23649, 23650,
       23651, 23652, 23653, 23654, 23655, 23656, 23657, 23658, 23659,
       23660, 23661, 23662, 23663, 23664, 23665, 23666, 23667, 23668,
       23669, 23670, 23671, 23672, 23673, 23674, 23675, 23676, 23677,
       23678, 23679

In [14]:
def seq_padding(X, padding=PAD):
    L = [len(x) for x in X]
    ML = max(L)
    padded_seq = np.array([np.concatenate([x,
                   [padding] * (ML - len(x))])
        if len(x) < ML else x for x in X])
    return padded_seq


## Batch class
The Batch class receives two inputs: src, which is the sequence of indices representing the tokens in the German sentence and trg, the sequence of indices for the tokens in the English translation.

The purpose of trg_mask is to conceal the subsequent tokens in the input, ensuring that the model relies solely on previous tokens for making predictions

In [15]:
class Batch:
    def __init__(self, src, trg=None, pad=0):
        src = torch.from_numpy(src).to(DEVICE).long()
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)  # Create a source mask to hide padding at the end of the sentence
        if trg is not None:
            trg = torch.from_numpy(trg).to(DEVICE).long()
            self.trg = trg[:, :-1]  # Create input to the decoder
            self.trg_y = trg[:, 1:]  # Shift the input one token to the right and use it as output of the decoder
            self.trg_mask = make_std_mask(self.trg, pad)  # Create a target mask 
            self.ntokens = (self.trg_y != pad).data.sum()
#A 



In [88]:
from utils.transformer_util import Batch

import numpy as np
def subsequent_mask(size):
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape),
                              k=1).astype('uint8')
    output = torch.from_numpy(subsequent_mask) == 0
    return output


def make_std_mask(tgt, pad):
    tgt_mask=(tgt != pad).unsqueeze(-2)
    output=tgt_mask & subsequent_mask(\
        tgt.size(-1)).type_as(tgt_mask.data)
    return output 

batches=[]
for b in batch_indexs:
    batch_en=[out_en_ids[x] for x in b]
    batch_de=[out_de_ids[x] for x in b]
    batch_en=seq_padding(batch_en)
    batch_de=seq_padding(batch_de)
    batches.append(Batch(batch_de,batch_en))
print(batches[2].trg)

tensor([[   3,   19, 1236,  ...,    0,    0,    0],
        [   3,  988,   11,  ...,    0,    0,    0],
        [   3,   19,  784,  ...,    0,    0,    0],
        ...,
        [   3,    6,   16,  ...,    0,    0,    0],
        [   3,    6,   12,  ...,    4,    0,    0],
        [   3,    6,   25,  ...,    8,  212,    5]])


In [89]:
src_vocab = len(de_word_dict)
tgt_vocab = len(en_word_dict)
print(f"there are {src_vocab} distinct German tokens")
print(f"there are {tgt_vocab} distinct English tokens")


there are 19214 distinct German tokens
there are 10837 distinct English tokens


## Input embedding from word embedding and positional encoding

Word embedding transforms tokens into dense, low-dimensional vectors that capture semantic relationships among them. These embeddings place similar words closer  in vector space, making them more efficient for machine learning models to process compared to sparse one-hot encodings. Moreover, the embeddings are learned directly from the training dataset during the model’s training process

In [21]:
from torch import nn
import math
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
    def forward(self, x):
        out = self.lut(x) * math.sqrt(self.d_model)
        return out


The PositionalEncoding() class generates vectors for sequence positions using sine functions for even indices 
and cosine functions for odd indices. One of the benefit sof using these trigonometric functions is that their outputs range between –1 and 1. Additionally, it’s important to note that requires_grad(False) means there’s no needto train these values. They remain constant across all inputs, and they don’t changeduring training.


In [22]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):  #A
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=DEVICE)
        position = torch.arange(0., max_len,
                                device=DEVICE).unsqueeze(1)
        div_term = torch.exp(torch.arange(
            0., d_model, 2, device=DEVICE)
            * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos)  #B
        pe[:, 1::2] = torch.cos(pe_pos)  #C
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)  #D
        out = self.dropout(x)
        return out
#A Initiate the class, allowing a maximum of 5000 positions
#B Apply sine function to even indexes in the vector
#C Apply cosine function to odd indexes in the vector
#D Add positional encoding to word embedding


## Encoder–decoder transformer

The attention() function takes query, key, and value as inputs and calculates attention and attention weights 

In [92]:
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query,
              key.transpose(-2, -1)) / math.sqrt(d_k)  #A
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)  #B
    p_attn = nn.functional.softmax(scores, dim=-1)  #C
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn  #D
#A Scaled attention score is the dot product of query and key, scaled by the square root of dk
#B If there is a mask, hide future elements in the sequence
#C Calculate attention weights
#D Return both attention and attention weights


In [93]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder,
                 src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder  # Define an encoder in the transformer
        self.decoder = decoder  # Define a decoder in the transformer
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt),
                            memory, src_mask, tgt_mask)
    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)  # Source language is encoded into an abstract vector representation by the encoder
        output = self.decode(memory, src_mask, tgt, tgt_mask)  # The decoder uses the vector representation to generate the translation in the target language
        return output


The encoder compresses a German sentence (e.g., “Wie geht es dir”) into a vector representation that captures its meaning. 

In [90]:
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)]) # N is the number of layers
        self.norm = LayerNorm(layer.size)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
            output = self.norm(x)
        return output


The decoder generates the English translation. The decoder then takes the output from the encoder and generates the translation in an autoregressive fashion: it generates one element at a time, based on the previously generated elements and the output from the encoder.

In [26]:
class Decoder(nn.Module):
    def __init__(self, layer, N): ## N: number of layers
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)]) 
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        output = self.norm(x)
        return output
    
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn,
                 feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([deepcopy(
        SublayerConnection(size, dropout)) for i in range(3)])
    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.sublayer[0](x, lambda x:
                 self.self_attn(x, x, x, tgt_mask))  # masked multi-head attention layer
        x = self.sublayer[1](x, lambda x:
                 self.src_attn(x, memory, memory, src_mask))  # cross-attention layer between the two languages
        output = self.sublayer[2](x, self.feed_forward)  # feed-forward network
        return output



## Creating a model to translate German to English

The create_model() function uses the Transformer() class we defined earlier, withfive essential elements: encoder, decoder, src_embed, tgt_embed, and generator.
Generator() class generates the most likely next token.The output of the Generator() class is the probability distribution in the target languagevocabulary.

In [27]:
def create_model(src_vocab, tgt_vocab, N, d_model,
                 d_ff, h, dropout=0.1):
    attn=MultiHeadedAttention(h, d_model).to(DEVICE)
    ff=PositionwiseFeedForward(d_model, d_ff, dropout).to(DEVICE)
    pos=PositionalEncoding(d_model, dropout).to(DEVICE)
    model = Transformer(
        Encoder(EncoderLayer(d_model,deepcopy(attn),deepcopy(ff),
                             dropout).to(DEVICE),N).to(DEVICE),  #A
        Decoder(DecoderLayer(d_model,deepcopy(attn),
             deepcopy(attn),deepcopy(ff), dropout).to(DEVICE),
                N).to(DEVICE),  #B
        nn.Sequential(Embeddings(d_model, src_vocab).to(DEVICE),
                      deepcopy(pos)),  #C
        nn.Sequential(Embeddings(d_model, tgt_vocab).to(DEVICE),
                      deepcopy(pos)),  #D
        Generator(d_model, tgt_vocab)).to(DEVICE)  #E
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model.to(DEVICE)
#A Create an encoder by instantiating the Encoder() class
#B Create a decoder by instantiating the Decoder() class
#C Create src_embed by generating input embeddings for the source language
#D Create tgt_embed by generating input embeddings for the target language
#E Create a generator by instantiating the Generator() class


N=6 layers, embedding dimension = 256, 8 heads

In [32]:
from utils.transformer_util import create_model

model = create_model(src_vocab, tgt_vocab, N=6,
    d_model=256, d_ff=1024, h=8, dropout=0.1)


# Translator function

In [34]:

def de2en(ger):
    tokenized_ger= [tok.text for tok in de_tokenizer.tokenizer(ger)]
    tokenized_ger=["BOS"]+tokenized_ger+["EOS"]
    geridx=[de_word_dict.get(i,UNK) for i in tokenized_ger]
    src=torch.tensor(geridx).long().to(DEVICE).unsqueeze(0)
    src_mask=(src!=0).unsqueeze(-2)
    memory=model.encode(src,src_mask)  # Use encoder to convert German to vector representations
    start_symbol=en_word_dict["BOS"]
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)
    translation=[]
    for i in range(100):
        out = model.decode(memory,src_mask,ys,
        subsequent_mask(ys.size(1)).type_as(src.data))
        prob = model.generator(out[:, -1])  # Predict the next English token using the decoder
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.ones(1, 1).type_as(
            src.data).fill_(next_word)], dim=1)
        sym = en_idx_dict[ys[0, -1].item()]
        if sym != 'EOS':  # Stops translating when the next token is EOS
            translation.append(sym)
        else:
            break
    trans=" ".join(translation)
    for x in '''?:;.,'("-!&)%''':
        trans=trans.replace(f" {x}",f"{x}")  # Join the predicted tokens to form an English sentence as the translation
    return trans


# Training

In [33]:
from utils.transformer_util import (NoamOpt, LabelSmoothing,
       SimpleLossCompute)
import torch

optimizer = NoamOpt(256, 1, 2000, torch.optim.Adam(
    model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))
criterion = LabelSmoothing(tgt_vocab,
                           padding_idx=0, smoothing=0.0)
loss_func = SimpleLossCompute(
            model.generator, criterion, optimizer)


In [35]:

model.eval()
print("BEFORE TRAINING")
for i in range(5):
    print("original Ger:", trainde[100+i])
    print("original Eng:", trainen[100+i])
    print("translated Eng:", de2en(trainde[100+i]))
    print("--\n")

for epoch in range(50):
    model.train()
    tloss=0
    tokens=0
    for batch in batches:
        out = model(batch.src, batch.trg,
                    batch.src_mask, batch.trg_mask)  #  Predict the next token using the transformer 
        loss = loss_func(out, batch.trg_y, batch.ntokens)  #Calculate loss and adjust model parameters 
        tloss += loss
        tokens += batch.ntokens  # Count the number of tokens in the batch
    print(f"Epoch {epoch}, average loss: {tloss/tokens}")
    model.eval()
    for i in range(5):
        print("original Ger:", trainde[100+i])
        print("original Eng:", trainen[100+i])
        print("translated Eng:", de2en(trainde[100+i]))
        print("--\n")
torch.save(model.state_dict(),"/content/txt2img/files/de2en.pth")  # Save the weights in the trained model after training



original Ger: Männliches Kleinkind in einem roten Hut, das sich an einem Geländer festhält.
original Eng: Toddler boy in a red hat holding on to some railings.
translated Eng: tumble tumble tumble tumble plying tumble clean clean clean clean twin twin twin twin twin twin twin twin twin twin clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean clean twin twin clean clean clean twin twin twin twin
--

original Ger: Drei Hunde stehen auf einer Wiese und eine Person kniet in der Nähe.
original Eng: Three dogs stand in a grassy field while a person kneels nearby.
translated Eng: twin tumble tumble plying twin mermaid twin mermaid 

KeyboardInterrupt: 

## Final assessment

In [ ]:

model.eval()
for i in range(5):
    print("original Ger:", trainde[100+i])
    print("original Eng:", trainen[100+i])
    print("translated Eng:", de2en(trainde[100+i]))


original Ger: Männliches Kleinkind in einem roten Hut, das sich an einem Geländer festhält.
original Eng: Toddler boy in a red hat holding on to some railings.
translated Eng: Toddler boy in a red hat holding on to some railings.
original Ger: Drei Hunde stehen auf einer Wiese und eine Person kniet in der Nähe.
original Eng: Three dogs stand in a grassy field while a person kneels nearby.
translated Eng: Three dogs stand in a grassy field and one person kneels nearby.
original Ger: Ein Mann steht vor einem Hochhaus.
original Eng: A man is standing in front of a skyscraper
translated Eng: A man is standing in front of a skyscraper
original Ger: Eine Frau fährt ihr Baby in einem Sportwagen im örtlichen Park spazieren.
original Eng: A woman is walking her baby with a stroller at the local park.
translated Eng: A woman is walking her baby with a stroller at the local park.
original Ger: Ein Mann in einem roten Hemd sitzt neben Obst, das zu verkaufen ist.
original Eng: A man in a red shirt 